# Medicina de Grupo Filter — North Region Breast Cancer Base

## Purpose and connection to the business problem

`NM_MODALIDADE_Medicina De Grupo` is a one-hot flag created during data treatment (notebook 04) from the health-insurance operator's institutional modality. It identifies rows billed by "Medicina de Grupo" operators, one of six mutually exclusive modalities in the base (the others being Autogestão, Cooperativa Médica, Cooperativa Odontológica, Filantropia and Seguradora Especializada em Saúde).

This notebook applies that flag as a **filter** and derives a **verticalization flag** from it. The filtered population is now the project's official modeling population: every notebook from `09_activity_matrix_operator.ipynb` onward reads `north_region_mama_2025_data_en_medicina_de_grupo.parquet` produced here, not the full treated base, so the operator-level and operator-municipality K-Means (09, 11), the feature/target matrix (12) and the three supervised candidates (14, 15, 16) are all fit on Medicina de Grupo operators. Section 4.3.2.8 of the documentation records this decision and its consequences. This notebook itself stays non-destructive: it reads `north_region_mama_2025_data_en.parquet` and never modifies it, and notebooks 01-07 and 14 are unaffected -- see their own scope notes.


## 1. Input and reproducible loading

**Upstream:** `04_data_treatment.ipynb`. **Required input:** `data/processed/north_region_mama_2025_data_en.parquet`. **Output:** two artifacts — `data/processed/north_region_mama_2025_data_en_medicina_de_grupo.parquet` (Medicina de Grupo rows only) and `data/processed/north_region_mama_2025_verticalized_operator_lookup.parquet` (per-`ID_PLANO` verticalization flag) — the original treated base is never modified. **Downstream:** `north_region_mama_2025_data_en_medicina_de_grupo.parquet` is the official input for `notebooks/09_activity_matrix_operator.ipynb` through `17_random_forest.ipynb` (14 excepted, see its own notebook); `scripts/run_pipeline.py` runs this notebook right before `09_activity_matrix_operator.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

INPUT_PATH = PROCESSED_DIR / "north_region_mama_2025_data_en.parquet"
OUTPUT_PATH = PROCESSED_DIR / "north_region_mama_2025_data_en_medicina_de_grupo.parquet"
VERTICALIZATION_LOOKUP_PATH = PROCESSED_DIR / "north_region_mama_2025_verticalized_operator_lookup.parquet"

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        "Missing treated dataset. Run 04_data_treatment.ipynb first. "
        f"Expected path: {INPUT_PATH}"
    )

# The six mutually exclusive one-hot flags produced by get_dummies(columns=["NM_MODALIDADE", ...])
# in notebook 04. Named here so the mutual-exclusivity check in Section 3 is self-contained.
MODALIDADE_COLUMNS = [
    "NM_MODALIDADE_Autogestão",
    "NM_MODALIDADE_Cooperativa Médica",
    "NM_MODALIDADE_Cooperativa Odontológica",
    "NM_MODALIDADE_Filantropia",
    "NM_MODALIDADE_Medicina De Grupo",
    "NM_MODALIDADE_Seguradora Especializada Em Saúde",
]
TARGET_COLUMN = "NM_MODALIDADE_Medicina De Grupo"

# All columns, not a subset: OUTPUT_PATH is now the official input for every notebook from
# 09_activity_matrix_operator.ipynb onward, so it has to carry the same columns as the general treated
# base (CD_MUNICIPIO_*, CD_PROCEDIMENTO, treatment_line_primary, PORTE, ...), not just the
# handful this notebook's own row-count and verticalization checks need.
df = pd.read_parquet(INPUT_PATH)
missing_columns = sorted(set(MODALIDADE_COLUMNS) - set(df.columns))
if missing_columns:
    raise KeyError(f"Required columns are missing: {missing_columns}")

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {INPUT_PATH}")

Loaded 310,340 rows and 119 columns from C:\Users\Inteli\Desktop\g02\data\processed\north_region_mama_2025_data_en.parquet


## 2. Apply the filter

`NM_MODALIDADE_Medicina De Grupo == 1` selects the rows billed by Medicina de Grupo operators. The filter is applied to an in-memory copy; `df` (the loaded treated base) is never mutated, and the on-disk treated base is untouched regardless of what this notebook does.

In [2]:
n_before = len(df)
df_medicina_de_grupo = df[df[TARGET_COLUMN] == 1].copy()
n_after = len(df_medicina_de_grupo)

print(f"Rows before filter: {n_before:,}")
print(f"Rows after filter (Medicina de Grupo only): {n_after:,} ({n_after / n_before:.2%} of the treated base)")
print(f"Distinct plans (ID_PLANO): {df_medicina_de_grupo['ID_PLANO'].nunique():,}")
print(f"Provider states covered: {sorted(df_medicina_de_grupo['UF_PRESTADOR'].unique())}")
print(f"Period covered: {df_medicina_de_grupo['MES'].min().date()} to {df_medicina_de_grupo['MES'].max().date()}")

subset_volume = df_medicina_de_grupo["QT_ITEM_EVENTO_INFORMADO"].sum()
total_volume = df["QT_ITEM_EVENTO_INFORMADO"].sum()
print(f"Volume (QT_ITEM_EVENTO_INFORMADO) in subset: {subset_volume:,.0f} "
      f"({subset_volume / total_volume:.2%} of the treated base's total volume)")

Rows before filter: 310,340
Rows after filter (Medicina de Grupo only): 70,004 (22.56% of the treated base)
Distinct plans (ID_PLANO): 1,052
Provider states covered: ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO']
Period covered: 2025-01-01 to 2025-12-01
Volume (QT_ITEM_EVENTO_INFORMADO) in subset: 822,851 (22.58% of the treated base's total volume)


## 3. Mutual-exclusivity check

The `NM_MODALIDADE_*` columns come from a single `pd.get_dummies` call in notebook 04, so exactly one flag should be set per row. This is verified on the full base and, specifically for the acceptance criteria of this filter, on the retained rows: no row kept by the filter should carry any other modality flag.

In [3]:
row_modality_sum = df[MODALIDADE_COLUMNS].sum(axis=1)
print(f"Rows in the full treated base with modality-flag sum != 1: {(row_modality_sum != 1).sum()}")

other_modality_columns = [c for c in MODALIDADE_COLUMNS if c != TARGET_COLUMN]
violations = int((df_medicina_de_grupo[other_modality_columns].sum(axis=1) > 0).sum())
print(f"Retained rows with another NM_MODALIDADE_* flag also set: {violations}")

assert violations == 0, "Mutual exclusivity violated: some retained rows carry more than one modality flag."
print("Mutual exclusivity holds: every retained row belongs to Medicina de Grupo only.")

Rows in the full treated base with modality-flag sum != 1: 0
Retained rows with another NM_MODALIDADE_* flag also set: 0
Mutual exclusivity holds: every retained row belongs to Medicina de Grupo only.


## 4. Save the filtered artifact

The filtered subset is written to its own file, separate from the treated base, so the original remains the single source of truth for the registered models. Re-running this notebook against an unchanged treated base reproduces the same output.

In [4]:
df_medicina_de_grupo.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved {len(df_medicina_de_grupo):,} rows to: {OUTPUT_PATH}")
print(f"Original treated base unchanged: {INPUT_PATH}")
print(f"Original treated base row count (unaffected by this notebook): {len(df):,}")

Saved 70,004 rows to: C:\Users\Inteli\Desktop\g02\data\processed\north_region_mama_2025_data_en_medicina_de_grupo.parquet
Original treated base unchanged: C:\Users\Inteli\Desktop\g02\data\processed\north_region_mama_2025_data_en.parquet
Original treated base row count (unaffected by this notebook): 310,340


## 5. Verticalization flag (partner-confirmed rule)

**Rule and its source.** The project partner confirmed that Medicina de Grupo operators are verticalized (they own their own care-provider network), while the project's earlier attempt to derive verticalization from data alone (Section 4.3.5 of the documentation, notebook 14) found it **could not be identified from non-hospital market share or concentration patterns** and left `VERTICALIZED_OPERATOR` as `Unknown`/`NA`, pending operator-provider ownership evidence.

This flag is **not** that evidence-backed label. It is a coarser, partner-confirmed rule applied to the regulatory modality classification: `VERTICALIZED_OPERATOR = 1` when `NM_MODALIDADE_Medicina De Grupo == 1`, else `0`. It is computed at the `ID_PLANO` level (modality is constant within a plan across every row — verified below) and saved as its own lookup artifact, not merged into the treated base or the modeling matrix, so it stays clearly separable from the `Unknown` evidence-backed field reserved in Section 4.3.5's pending-fields table.

In [5]:
per_plan_modalities = df.groupby("ID_PLANO")[MODALIDADE_COLUMNS].nunique()
inconsistent_plans = int((per_plan_modalities.eq(1).all(axis=1) == False).sum())
print(f"Plans with more than one modality across rows: {inconsistent_plans} (modality is constant per ID_PLANO)")
assert inconsistent_plans == 0, "A plan changes modality across rows; the per-plan flag below would be ambiguous."

per_plan = df.groupby("ID_PLANO")[MODALIDADE_COLUMNS].max().reset_index()
per_plan["VERTICALIZED_OPERATOR"] = per_plan[TARGET_COLUMN].astype(int)
verticalization_lookup = per_plan[["ID_PLANO", "VERTICALIZED_OPERATOR"]]

print(f"Distinct ID_PLANO: {len(verticalization_lookup):,}")
print(f"VERTICALIZED_OPERATOR = 1 (Medicina de Grupo): {(verticalization_lookup['VERTICALIZED_OPERATOR'] == 1).sum():,} plans")
print(f"VERTICALIZED_OPERATOR = 0 (all other modalities): {(verticalization_lookup['VERTICALIZED_OPERATOR'] == 0).sum():,} plans")

verticalization_lookup.to_parquet(VERTICALIZATION_LOOKUP_PATH, index=False)
print(f"Saved to: {VERTICALIZATION_LOOKUP_PATH}")

Plans with more than one modality across rows: 0 (modality is constant per ID_PLANO)
Distinct ID_PLANO: 3,289
VERTICALIZED_OPERATOR = 1 (Medicina de Grupo): 1,052 plans
VERTICALIZED_OPERATOR = 0 (all other modalities): 2,237 plans
Saved to: C:\Users\Inteli\Desktop\g02\data\processed\north_region_mama_2025_verticalized_operator_lookup.parquet


## Summary

- This notebook is non-destructive by itself: `north_region_mama_2025_data_en.parquet` (the general treated base) is untouched. What changed is downstream -- `north_region_mama_2025_data_en_medicina_de_grupo.parquet` is now the official input to notebooks 09-17 (14 excepted), replacing the full population there.
- `north_region_mama_2025_data_en_medicina_de_grupo.parquet` holds 70,004 of 310,340 rows (22.56%), 1,052 of 3,289 plans (32.0%).
- `north_region_mama_2025_verticalized_operator_lookup.parquet` is a separate per-`ID_PLANO` lookup (1,052 verticalized, 2,237 non-verticalized, of 3,289 plans), keyed for joining into a future demand-transfer analysis. Its rule is the partner's domain confirmation, not the operator-provider ownership evidence Section 4.3.5 originally required — that distinction should stay documented wherever this lookup is used.
- Mutual exclusivity of the `NM_MODALIDADE_*` one-hot group was verified on both the full base and the filtered subset (0 violations).